# Tokenization


- **What it does**: Turns text into numbers for models.

- **Why it’s done**: Compresses text smartly, like zipping data.

- **Demo**: Using 'The cat is mad' to show token merging.

## Step 1: Initialize Vocabulary

- **What**: Split text into characters, add `</w>` for word ends.

- **Think of it**: Like breaking words into individual letters.

- **Goal**: Create a starting vocabulary.

In [1]:
import collections
import re

def initialize_vocabulary(text):
    vocab = collections.defaultdict(int)
    words = text.strip().split()
    for word in words:
        vocab[' '.join(list(word)) + ' </w>'] += 1  # Split word into chars + </w>
    return vocab

# Toy text
text = 'The cat is mad'
vocab = initialize_vocabulary(text)
print('Initial vocabulary:', {k.replace(' ', '  '): v for k, v in dict(vocab).items()})

Initial vocabulary: {'T  h  e  </w>': 1, 'c  a  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}


## Step 2: Find and Merge Frequent Pairs

- **Process**: Count adjacent character pairs, merge the most frequent into a single token.

- **Analogy**: Like combining common syllables to simplify words.

- **Why**: Builds larger tokens to streamline vocabulary.

In [2]:
def get_pairs_and_counts(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()  # Split word into tokens
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq  # Count adjacent pairs
    return pairs

def merge_pair_in_vocabulary(pair, vocab_in):
    vocab_out = {}
    bigram = re.escape(' '.join(pair))  # Escape spaces in pair
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')  # Match pair
    for word in vocab_in:
        word_out = p.sub(''.join(pair), word)  # Merge pair
        vocab_out[word_out] = vocab_in[word]
    return vocab_out

## Step 3: Run Tokenization

- **What**: Iteratively merge the most frequent pair, updating the vocabulary.

- **How**: Show each merge step and the resulting vocabulary.

- **Insight**: Like decoding patterns to simplify text data.

In [3]:
def tokenize(text, num_merges):
    vocab = initialize_vocabulary(text)  # Start with character vocab
    for i in range(num_merges):
        pairs = get_pairs_and_counts(vocab)  # Find frequent pairs
        if not pairs:
            break
        most_frequent_pair = max(pairs, key=pairs.get)  # Pick top pair
        vocab = merge_pair_in_vocabulary(most_frequent_pair, vocab)  # Update vocab
        formatted_vocab = {k.replace(' ', '  '): v for k, v in dict(vocab).items()}
        print(f'Merge {i+1}: Pair {most_frequent_pair} -> Vocabulary: {formatted_vocab}')
    tokens = collections.defaultdict(int)
    for word, freq in vocab.items():
        for token in word.split():
            tokens[token] += freq  # Count final tokens
    return tokens, vocab

# Run for 11 merges
print('Initial vocabulary:', {k.replace(' ', '  '): v for k, v in dict(vocab).items()})
tokens, vocab = tokenize(text, num_merges=11)
print('Final tokens:', {k: v for k, v in dict(tokens).items()})

Initial vocabulary: {'T  h  e  </w>': 1, 'c  a  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 1: Pair ('T', 'h') -> Vocabulary: {'Th  e  </w>': 1, 'c  a  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 2: Pair ('Th', 'e') -> Vocabulary: {'The  </w>': 1, 'c  a  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 3: Pair ('The', '</w>') -> Vocabulary: {'The</w>': 1, 'c  a  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 4: Pair ('c', 'a') -> Vocabulary: {'The</w>': 1, 'ca  t  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 5: Pair ('ca', 't') -> Vocabulary: {'The</w>': 1, 'cat  </w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 6: Pair ('cat', '</w>') -> Vocabulary: {'The</w>': 1, 'cat</w>': 1, 'i  s  </w>': 1, 'm  a  d  </w>': 1}
Merge 7: Pair ('i', 's') -> Vocabulary: {'The</w>': 1, 'cat</w>': 1, 'is  </w>': 1, 'm  a  d  </w>': 1}
Merge 8: Pair ('is', '</w>') -> Vocabulary: {'The</w>': 1, 'cat</w>': 1, 'is</w>': 1, 'm  a  d  </w>': 1}
Merge 9: Pair 

## Wrap-Up

- **Result**: From individual characters to tokens like `The`, `mad` for efficient text representation.

- **Why it matters**: Prepares text for transformers to process.

